# Module 3 — Performance

**What it is:** Performance tuning measures bottlenecks (profiling), reduces repeated work (caching), picks the right data structures, and chooses threading vs multiprocessing vs asyncio based on workload type.

**Where we use it:** Data pipelines, APIs under load, numerical code, and any system where latency or throughput is a requirement.

**Why we use it:** Understanding *where* time is spent prevents premature optimization and guides effective improvements.

## Timing with `timeit` and `perf_counter`

In [ ]:
import timeit
import time

# timeit runs code many times for stable measurement
list_time = timeit.timeit("sum(range(1000))", number=10_000)
gen_time = timeit.timeit("sum(x for x in range(1000))", number=10_000)
print(f"sum(range):     {list_time:.4f}s")
print(f"sum generator:  {gen_time:.4f}s")

start = time.perf_counter()
_ = [x * x for x in range(100_000)]
print(f"List comp:      {time.perf_counter() - start:.4f}s")

## Caching with `lru_cache`

In [ ]:
from functools import lru_cache
import time

@lru_cache(maxsize=None)
def fib(n: int) -> int:
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

def fib_no_cache(n: int) -> int:
    if n < 2:
        return n
    return fib_no_cache(n - 1) + fib_no_cache(n - 2)

n = 30
start = time.perf_counter()
result = fib(n)
cached_time = time.perf_counter() - start

start = time.perf_counter()
fib_no_cache(n)
uncached_time = time.perf_counter() - start

print(f"fib({n}) = {result}")
print(f"With cache:    {cached_time:.4f}s")
print(f"Without cache: {uncached_time:.4f}s")
print(f"Cache info:    {fib.cache_info()}")

## Profiling with `cProfile`

In [ ]:
import cProfile
import pstats
from io import StringIO

def slow_function():
    total = 0
    for i in range(50_000):
        total += i ** 2
    return total

profiler = cProfile.Profile()
profiler.enable()
slow_function()
profiler.disable()

stream = StringIO()
stats = pstats.Stats(profiler, stream=stream).sort_stats("cumulative")
stats.print_stats(5)
print(stream.getvalue())

## Choosing the right concurrency model

In [ ]:
# Quick reference — pick based on workload:
#
# | Workload   | Best tool        | Reason                          |
# |------------|------------------|---------------------------------|
# | CPU-bound  | multiprocessing  | Bypasses GIL, uses all cores    |
# | I/O-bound  | threading/async  | Overlap waiting on network/disk |
# | Many conn  | asyncio          | Low memory vs thousands threads |
#
# Also consider: list vs set lookups, generators for large data,
# __slots__ for many small objects, and numpy for numeric arrays.

LOOKUPS = {"python", "java", "go"}
print("'python' in set:", "python" in LOOKUPS)  # O(1) average

big = range(1_000_000)
gen_sum = sum(x for x in big if x % 2 == 0)
print("Generator filter sum (sample):", gen_sum)